# AYADEN Lina

In this notebook, there is the code for question 2,3,5,7 and 8.
The code for question 7 and 8 has been merged.



## Question 2

We constructed a "Time-Expanded Graph". It means we duplicate every office for every phase (Phase 0, Phase 1, etc.) and each node in the graph represents a specific office at a specific phase $(o, p)$.

Then, finding the minimum moves is just finding the Shortest Path in this graph. We used a dynamic programming algorithm to calculate the cost. It is very fast because the graph is a Directed Acyclic Graph, so we calculate phase by phase. If a wing is closed, we just put the cost to infinity

In [53]:
import numpy as np

def q2():
    # Initial setup
    phases = 6
    offices = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
    off_idx = {name: i for i, name in enumerate(offices)}
    n_off = len(offices)

    # Building a mask to block out closed offices during renovations
    closed_mask = np.zeros((phases, n_off), dtype=bool)
    schedule = {1: ['B1','B2','B3'], 2: ['D1','D2','D3'], 3: ['C1','C2'], 4: ['A1','A2']}

    # Block E1/E2 at start and then follow the renovation schedule
    for o in ['E1', 'E2']: closed_mask[0, off_idx[o]] = True
    for p, closed_list in schedule.items():
        for off in closed_list:
            closed_mask[p, off_idx[off]] = True

    # Where everyone starts in Phase 0
    init = {
        'P1':'A1', 'P2':'A1', 'P3':'A2', 'P4':'A2', 'S1':'B2', 'S2':'B2',
        'S3':'C2', 'S4':'C2', 'T1':'C1', 'T2':'C1', 'T3':'D1', 'T4':'B3',
        'M1':'D3', 'M2':'D3', 'M3':'D2', 'M4':'B1', 'O1':'B1', 'O2':'B3'
    }

    # Where everyone is supposed to end up in Phase 5
    targets = {
        'P1':'A1', 'P2':'A1', 'P3':'A2', 'P4':'A2', 'O1':'B1', 'O2':'D2',
        'M1':'B2', 'M2':'B2', 'M3':'B3', 'M4':'D2', 'T1':'C1', 'T2':'C1',
        'T3':'D1', 'T4':'E1', 'S1':'C2', 'S2':'C2', 'S3':'D3', 'S4':'E2'
    }

    # Tracking how many people are in each room per phase (Max 2)
    occupancy = np.zeros((phases, n_off), dtype=int)
    all_paths = {}

    # Iterate through each person to find their best path
    for person, start_off in sorted(init.items()):
        target = targets[person]
        dp = np.full((phases, n_off), np.inf) # Cost matrix
        parent = np.full((phases, n_off), -1, dtype=int) # For backtracking

        # Start at phase 0
        start_i = off_idx[start_off]
        dp[0, start_i] = 0
        occupancy[0, start_i] += 1

        # Look forward through the phases
        for t in range(phases - 1):
            for u in range(n_off):
                if dp[t, u] == np.inf: continue

                # Check every office to see where we can move next
                for v in range(n_off):
                    # Skip if the room is under renovation or already full (Cap 2)
                    if closed_mask[t+1, v] or occupancy[t+1, v] >= 2:
                        continue

                    # Move cost is 1, staying put is 0
                    cost = 0 if u == v else 1
                    new_cost = dp[t, u] + cost

                    # If this path is cheaper, save it
                    if new_cost < dp[t+1, v]:
                        dp[t+1, v] = new_cost
                        parent[t+1, v] = u

        # Backtrack from the target in Phase 5 to Phase 0
        curr = off_idx[target]

        # Fallback: if the target is full or closed, pick the next best room
        if dp[5, curr] == np.inf:
            curr = np.argmin(dp[5, :])

        path_idx = [curr]
        for t in range(5, 0, -1):
            prev = parent[t, curr]
            path_idx.append(prev)
            # Update the global occupancy so others know this seat is taken
            occupancy[t, curr] += 1
            curr = prev

        # Reverse path to get 0 -> 5 order
        path_idx.reverse()
        all_paths[person] = [offices[i] for i in path_idx]

    # Print out the results for each phase
    total_moves = 0
    for t in range(1, 6):
        print(f"\nPHASE {t}:")
        moved = False
        for p, path in sorted(all_paths.items()):
            # Only print if the person actually changed rooms
            if path[t] != path[t-1]:
                print(f"  {p:2}: {path[t-1]} -> {path[t]}")
                total_moves += 1
                moved = True
        if not moved: print("  No moves required.")

    print("-" * 30)
    print(f"Minimum moves is : {total_moves}")

if __name__ == "__main__":
    q2()


PHASE 1:
  M4: B1 -> A1
  O1: B1 -> A1
  O2: B3 -> A2
  P1: A1 -> E1
  P2: A1 -> E1
  P4: A2 -> E2
  S1: B2 -> E2
  S2: B2 -> C1
  T2: C1 -> D1
  T4: B3 -> D2

PHASE 2:
  M1: D3 -> B2
  M2: D3 -> B2
  M3: D2 -> B3
  S2: C1 -> B3
  T1: C1 -> B1
  T2: D1 -> B1
  T3: D1 -> C1
  T4: D2 -> C1

PHASE 3:
  S3: C2 -> D3
  S4: C2 -> D1
  T3: C1 -> D1
  T4: C1 -> D3

PHASE 4:
  M4: A1 -> D2
  O1: A1 -> B1
  O2: A2 -> D2
  P3: A2 -> B1
  S1: E2 -> C2
  T1: B1 -> C1
  T2: B1 -> C1

PHASE 5:
  P1: E1 -> A1
  P2: E1 -> A1
  P3: B1 -> A2
  P4: E2 -> A2
  S2: B3 -> C2
  S4: D1 -> E2
  T4: D3 -> E1
------------------------------
Minimum moves is : 36


#Question 3:

For this part, the logic is similar to Question 2, but we add a Separation Constraint . The rule is that Students (S) and Presidency (P) cannot be neighbors.In our model, the cost of being in an office now depends on who is next door. If a President is in $A2$, a Student cannot be in $D3$ or $B1$ because they share an edge in the building's adjacency graph.To solve this, we used a Las Vegas algorithm. The code tries different seating orders for each phase until it finds one where everyone fits without violating the safety rule. It prioritizes the Presidency and Students to ensure they get valid spots before the other occupants fill the remaining capacity.One specific conflict appeared in Phase 5: the target for a President is $A1$ and for a Student is $D3$. Since these offices are neighbors, the math proves they cannot both occupy their targets . The algorithm resolves this by keeping the Student in a safe office nearby to respect the hard constraint.

In [54]:
import random

def q3():
    # initial setup
    OFFICES = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
    CAPACITY = 2

    # how rooms are connected for adjacency checks
    ADJACENCY = {
        'A1': ['A2'], 'A2': ['A1', 'B1', 'D3'], 'B1': ['A2', 'B2', 'D2'], 'B2': ['B1', 'B3'],
        'B3': ['B2', 'C1'], 'C1': ['B3', 'C2', 'D1'], 'C2': ['C1'],
        'D1': ['C2', 'D2', 'C1'], 'D2': ['D1', 'D3', 'B1'],
        'D3': ['D2', 'A2', 'E1'], 'E1': ['D3', 'E2'], 'E2': ['E1']
    }

    # which offices are closed in which phase
    CLOSED = {1: ['B1', 'B2', 'B3'], 2: ['D1', 'D2', 'D3'], 3: ['C1', 'C2'], 4: ['A1', 'A2'], 5: []}

    # where everyone starts at phase 0
    init = {
        'P1': 'A1', 'P2': 'A1', 'P3': 'A2', 'P4': 'A2', 'S1': 'B2', 'S2': 'B2', 'S3': 'C2', 'S4': 'C2',
        'T1': 'C1', 'T2': 'C1', 'T3': 'D1', 'T4': 'B3', 'M1': 'D3', 'M2': 'D3', 'M3': 'D2', 'M4': 'B1',
        'O1': 'B1', 'O2': 'B3'
    }

    # where everyone wants to go at the end
    targets = {
        'P1': 'A1', 'P2': 'A1', 'P3': 'A2', 'P4': 'A2', 'O1': 'B1', 'O2': 'D2', 'M1': 'B2', 'M2': 'B2',
        'M3': 'B3', 'M4': 'D2', 'T1': 'C1', 'T2': 'C1', 'T3': 'D1', 'T4': 'E1', 'S1': 'C2', 'S2': 'C2',
        'S3': 'D3', 'S4': 'E2'
    }

    # check if presidency and students are in adjacent rooms
    def is_safe(seat, p, m):
        if p[0] not in ['P', 'S']: return True
        # if you are P, you cant be near S (and vice versa)
        enemy = 'S' if p[0] == 'P' else 'P'
        for n in ADJACENCY.get(seat, []):
            for person, loc in m.items():
                if loc == n and person.startswith(enemy): return False
        return True

    # track current state and total moves
    state = init.copy()
    hist = []
    moves = 0

    # loop through the 5 phases
    for ph in range(1, 6):
        closed = CLOSED[ph]
        people = list(init.keys())
        old_state = state.copy()

        # try random shuffles to find a valid layout that fits everyone
        done = False
        for attempt in range(10000):
            nxt = {}
            occ = {o: 0 for o in OFFICES}
            random.shuffle(people)
            # prioritize P and S so they dont get blocked by others and fail adjacency
            people.sort(key=lambda x: 0 if x[0] == 'P' else (1 if x[0] == 'S' else 2))

            ok = True
            for p in people:
                prev, tgt = state[p], targets[p]

                # if final phase, try to hit the target room
                if ph == 5:
                    if tgt not in closed and is_safe(tgt, p, nxt): cands = [tgt]
                    else:
                        cands = [o for o in OFFICES if o not in closed]
                        cands.sort(key=lambda x: 0 if 'E' in x else 1) # use E wings if target fails
                else:
                    # try to stay put, then try target, then try E wings
                    cands = [o for o in OFFICES if o not in closed]
                    cands.sort(key=lambda x: 0 if x == prev else (1 if x == tgt else (2 if 'E' in x else 3)))

                # pick first available room that isnt full and is safe
                chosen = None
                for s in cands:
                    if occ[s] < CAPACITY and is_safe(s, p, nxt):
                        chosen = s
                        break

                if chosen:
                    nxt[p] = chosen
                    occ[chosen] += 1
                else:
                    ok = False
                    break

            # if everyone got a seat, save the phase
            if ok:
                state = nxt
                done = True
                break

        # stop if no solution found
        if not done: break

        # record the moves made this phase
        for p in sorted(people):
            if old_state[p] != state[p]:
                moves += 1
                msg = f"{old_state[p]} -> {state[p]}"
                if ph == 5 and state[p] != targets[p]: msg += " (Conflict)"
                hist.append((ph, p, msg))

    # final printout
    print(f"Minimum moves is : {moves}")
    for ph_idx in range(1, 6):
        print(f"\nPHASE {ph_idx}:")
        found = False
        for h_ph, p, m in hist:
            if h_ph == ph_idx:
                print(f"  {p:2}: {m}")
                found = True
        if not found: print("  No moves.")

if __name__ == "__main__":
    q3()

Minimum moves is : 48

PHASE 1:
  M3: D2 -> D1
  M4: B1 -> D2
  O1: B1 -> E2
  O2: B3 -> D2
  S1: B2 -> C2
  S2: B2 -> E1
  S3: C2 -> E1
  T4: B3 -> E2

PHASE 2:
  M1: D3 -> B2
  M2: D3 -> B3
  M3: D1 -> B3
  M4: D2 -> B1
  O1: E2 -> B1
  O2: D2 -> E2
  T3: D1 -> E2
  T4: E2 -> B2

PHASE 3:
  M2: B3 -> D1
  O1: B1 -> B3
  O2: E2 -> D2
  S1: C2 -> E1
  S3: E1 -> E2
  S4: C2 -> E2
  T1: C1 -> D2
  T2: C1 -> B1
  T3: E2 -> D1

PHASE 4:
  M4: B1 -> D2
  O2: D2 -> B2
  P1: A1 -> E1
  P2: A1 -> E2
  P3: A2 -> E1
  P4: A2 -> E2
  S1: E1 -> C2
  S2: E1 -> C2
  S3: E2 -> B1
  S4: E2 -> B1
  T2: B1 -> C1
  T4: B2 -> C1

PHASE 5:
  M2: D1 -> B2
  O1: B3 -> B1
  O2: B2 -> D2
  P1: E1 -> A1
  P2: E2 -> A1
  P3: E1 -> A2
  P4: E2 -> A2
  S3: B1 -> E1 (Conflict)
  S4: B1 -> E2
  T1: D2 -> C1
  T4: C1 -> E1


##Comparing results in question 2 and question 3

Adding the Separation Constraint makes the total moves increase because Students and the Presidency are no longer allowed to be neighbors . While we found a fixed result of 36 moves for Question 2, the results for Question 3 vary between 41 and 52 (approximately) moves because the algorithm is non-deterministic . Since the building is often full during renovations, Students are forced to take longer detours to stay away from the Presidency.

##Question 5

Our algorithm uses a scoring system so the cost for each office is its move cost plus the penalty if it is not the occupant's final target.

In [55]:
import random

def q5():
    # room setup
    offices = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']
    limit = 2
    lmbda = 100 # penalty weight for being in the wrong office

    # renovation schedule from the project pdf
    closed = {
        1:['B1','B2','B3'], 2:['D1','D2','D3'],
        3:['C1','C2'], 4:['A1','A2'], 5:[]
    }

    # where everyone starts (phase 0)
    init = {
        'P1':'A1', 'P2':'A1', 'P3':'A2', 'P4':'A2', 'S1':'B2', 'S2':'B2',
        'S3':'C2', 'S4':'C2', 'T1':'C1', 'T2':'C1', 'T3':'D1', 'T4':'B3',
        'M1':'D3', 'M2':'D3', 'M3':'D2', 'M4':'B1', 'O1':'B1', 'O2':'B3'
    }

    # where everyone is ideally supposed to be
    targets = {
        'P1':'A1', 'P2':'A1', 'P3':'A2', 'P4':'A2', 'O1':'B1', 'O2':'D2',
        'M1':'B2', 'M2':'B2', 'M3':'B3', 'M4':'D2', 'T1':'C1', 'T2':'C1',
        'T3':'D1', 'T4':'E1', 'S1':'C2', 'S2':'C2', 'S3':'D3', 'S4':'E2'
    }

    best_score = 1e9
    best_log = []
    final_moves = 0

    # we run this 5000 times to find the plan with the lowest total penalty
    for _ in range(5000):
        curr = init.copy()
        log = []
        mv_count = 0
        dev_count = 0
        fail = False

        for ph in range(1, 6):
            cl = closed[ph]
            ppl = list(init.keys())

            # try different combinations
            random.shuffle(ppl)
            # prioritize people whose target room is actually open right now
            ppl.sort(key=lambda x: 0 if targets[x] not in cl else 1)

            nxt = {}
            occ = {o: 0 for o in offices}

            for p in ppl:
                old, tgt = curr[p], targets[p]
                options = []

                # check all rooms that aren't being renovated
                for s in [o for o in offices if o not in cl]:
                    if occ[s] >= limit: continue

                    # cost = 1 if they move, plus penalty if it's not their target room
                    c = (1 if s != old else 0)
                    if ph < 5:
                        if s != tgt: c += lmbda
                    else:
                        # final phase makes it very expensive to miss the target
                        if s != tgt: c += 9999

                    options.append((c, s))

                if not options:
                    fail = True; break

                # pick the room with the lowest penalty/move cost
                options.sort()
                best_s = options[0][1]
                nxt[p] = best_s
                occ[best_s] += 1

                # record moves
                if best_s != old:
                    mv_count += 1
                    log.append((ph, p, f"{old} -> {best_s}"))

            if fail: break

            # calculate deviation so how many people are in the wrong room this phase
            dev_count += sum(1 for p in ppl if nxt[p] != targets[p] and ph < 5)
            curr = nxt

        if not fail:
            # calculate the final score: moves + (penalty weight * total time spent in wrong rooms)
            score = mv_count + (lmbda * dev_count)
            if score < best_score:
                best_score, best_log, final_moves = score, log, mv_count

    # final results printout
    print(f"Minimum moves is : {final_moves}")
    for i in range(1, 6):
        print(f"\nPHASE {i}:")
        found = False
        for ph, p, m in best_log:
            if ph == i:
                print(f"  {p:2}: {m}")
                found = True
        if not found: print("  No moves.")
    print("-" * 30)
    print(f"Total penalty score: {best_score}")

if __name__ == "__main__":
    q5()

Minimum moves is : 38

PHASE 1:
  M4: B1 -> D2
  S1: B2 -> C2
  O2: B3 -> D2
  S4: C2 -> E2
  S2: B2 -> C2
  S3: C2 -> D3
  T4: B3 -> E1
  O1: B1 -> D1
  M3: D2 -> E1
  M2: D3 -> E2

PHASE 2:
  M2: E2 -> B2
  O1: D1 -> B1
  M3: E1 -> B3
  M1: D3 -> B2
  M4: D2 -> B1
  O2: D2 -> B3
  T3: D1 -> E1
  S3: D3 -> E2

PHASE 3:
  M4: B1 -> D2
  S3: E2 -> D3
  T3: E1 -> D1
  O2: B3 -> D2
  S2: C2 -> B1
  S1: C2 -> B3
  T2: C1 -> D1
  T1: C1 -> D3

PHASE 4:
  T2: D1 -> C1
  S2: B1 -> C2
  S1: B3 -> C2
  T1: D3 -> C1
  P1: A1 -> B1
  P4: A2 -> B3
  P3: A2 -> D1
  P2: A1 -> D3

PHASE 5:
  P4: B3 -> A2
  P2: D3 -> A1
  P3: D1 -> A2
  P1: B1 -> A1
------------------------------
Total penalty score: 1638


##Comparing results in question 2 and question 5

In question 5, the total moves increase from 36 (in question 2) to 38 because the high penalty ($\lambda = 100$) changes the goal. Instead of just trying to move as little as possible, the math now forces people to get into their offices as fast as they can . This means people might move into their target office early, even if they have to move out again later for a renovation, just to avoid the heavy penalty of being in the wrong place. The final score of 1638 includes these 38 moves and the cost of the time people spent away from their targets during the middle phases

##Question 7 and 8

we used a Semidefinite Programming (SDP) Relaxation to solve the move plan phase by phase . Instead of picking offices directly, the math turns each choice into a vector in a higher-dimensional space to find a better global fit . We applied heavy penalties for being in a closed office or missing a final target, then used a Randomized Rounding technique to turn those abstract vectors back into real office assignments while keeping each room under the 2-person limit

In [56]:
import numpy as np

def q7_q8():
    # 12 offices and 18 people to move
    offices = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'D3', 'E1', 'E2']

    # current seats at phase 0
    alloc = {
        'P1':'A1', 'P2':'A1', 'P3':'A2', 'P4':'A2', 'S1':'B2', 'S2':'B2',
        'S3':'C2', 'S4':'C2', 'T1':'C1', 'T2':'C1', 'T3':'D1', 'T4':'B3',
        'M1':'D3', 'M2':'D3', 'M3':'D2', 'M4':'B1', 'O1':'B1', 'O2':'B3'
    }

    # where everyone needs to be at the very end
    targets = {
        'P1':'A1', 'P2':'A1', 'P3':'A2', 'P4':'A2', 'S1':'B2', 'S2':'B2',
        'S3':'C2', 'S4':'C2', 'O1':'B1', 'O2':'B3', 'M1':'B1', 'M2':'D2',
        'M3':'D3', 'M4':'D3', 'T1':'B3', 'T2':'C1', 'T3':'C1', 'T4':'D1'
    }

    ppl = sorted(list(alloc.keys()))
    N_p, N_o = len(ppl), len(offices)
    dim = N_p * N_o # 216 total combinations

    # the plan: which rooms close in which phase
    schedule = [
        (1, ['B1', 'B2', 'B3'], False),
        (2, ['D1', 'D2', 'D3'], False),
        (3, ['C1', 'C2'], False),
        (4, ['A1', 'A2'], False),
        (5, [], True) # final phase forces everyone to target rooms
    ]

    total_moves = 0

    for ph, closed, end_goal in schedule:
        # Question7
        # building the cost matrix C for this phase
        C = np.zeros(dim)
        for i, p in enumerate(ppl):
            curr_off = alloc[p]
            for j, o in enumerate(offices):
                idx = i * N_o + j
                # penalty for being in a room under renovation
                if o in closed: C[idx] += 1000.0
                # cost of 1 for moving
                if o != curr_off: C[idx] += 1.0
                # extra penalty if we miss the target in phase 5
                if end_goal and o != targets[p]: C[idx] += 1000.0

        # solving the SDP by moving Z toward the lowest cost
        Z = np.eye(dim)
        for _ in range(200):
            # gradient step: push diagonal down based on cost
            np.fill_diagonal(Z, Z.diagonal() - 0.05 * C)
            # fix the diagonal back to 1
            np.fill_diagonal(Z, 1.0)
            # make sure Z stays positive semidefinite (no negative eigenvalues)
            vals, vecs = np.linalg.eigh(Z)
            vals[vals < 0] = 0
            Z = vecs @ np.diag(vals) @ vecs.T

        # Question 8
        # turn the fuzzy SDP matrix into scores we can actually use
        vals, vecs = np.linalg.eigh(Z)
        V = vecs @ np.diag(np.sqrt(np.maximum(vals, 0)))

        best_ph_alloc = {}
        best_ph_moves = 999

        # try 100 random directions to find a valid room layout
        for _ in range(100):
            # random projection
            r = np.random.normal(0, 1, dim)
            scores = V @ r

            cands = []
            for i, p in enumerate(ppl):
                for j, o in enumerate(offices):
                    s = scores[i * N_o + j]
                    # ignore rooms that are closed
                    if o in closed: s -= 10000
                    cands.append((s, p, o))

            # sort rooms by their high scores
            cands.sort(key=lambda x: x[0], reverse=True)

            tmp_alloc = {}
            counts = {off: 0 for off in offices}
            done = {p: False for p in ppl}

            # fill rooms while staying under the 2-person limit
            for _, p, o in cands:
                if not done[p] and counts[o] < 2 and o not in closed:
                    tmp_alloc[p] = o
                    counts[o] += 1
                    done[p] = True

            # if everyone got a seat, count the moves and save the best one
            if all(done.values()):
                mvs = sum(1 for p in ppl if tmp_alloc[p] != alloc[p])
                if mvs < best_ph_moves:
                    best_ph_moves = mvs
                    best_ph_alloc = tmp_alloc

        # print moves for this phase
        print(f"PHASE {ph}:")
        for p in ppl:
            old, new = alloc[p], best_ph_alloc.get(p, alloc[p])
            if old != new:
                print(f"  {p:2}: {old} -> {new}")

        print(f"Moves in Phase {ph}: {best_ph_moves}")
        total_moves += best_ph_moves
        alloc = best_ph_alloc

    print("-" * 30)
    print(f"Total Moves: {total_moves}")

if __name__ == "__main__":
    q7_q8()

PHASE 1:
  M1: D3 -> D2
  M2: D3 -> E1
  M4: B1 -> D3
  O1: B1 -> A2
  O2: B3 -> D1
  P1: A1 -> C1
  P3: A2 -> E2
  S1: B2 -> A1
  S2: B2 -> E2
  S4: C2 -> C1
  T1: C1 -> D3
  T2: C1 -> E1
  T4: B3 -> C2
Moves in Phase 1: 13
PHASE 2:
  M1: D2 -> E1
  M3: D2 -> C2
  M4: D3 -> C1
  O1: A2 -> B1
  O2: D1 -> B3
  P1: C1 -> A2
  P2: A1 -> E2
  P3: E2 -> B3
  P4: A2 -> B1
  S3: C2 -> A1
  T1: D3 -> B2
  T2: E1 -> A2
  T3: D1 -> B2
Moves in Phase 2: 13
PHASE 3:
  M1: E1 -> B1
  M2: E1 -> E2
  M3: C2 -> D3
  M4: C1 -> B2
  O1: B1 -> A2
  O2: B3 -> D1
  P4: B1 -> D2
  S2: E2 -> D3
  S4: C1 -> D1
  T2: A2 -> B3
  T3: B2 -> D2
  T4: C2 -> E1
Moves in Phase 3: 12
PHASE 4:
  M1: B1 -> D2
  M2: E2 -> E1
  O1: A2 -> B3
  P1: A2 -> E2
  P2: E2 -> B2
  P3: B3 -> B1
  S1: A1 -> C1
  S2: D3 -> C2
  S3: A1 -> D3
  T1: B2 -> C2
  T3: D2 -> B1
Moves in Phase 4: 11
PHASE 5:
  M1: D2 -> D1
  M2: E1 -> C1
  M3: D3 -> B2
  M4: B2 -> C2
  P1: E2 -> A1
  P2: B2 -> D2
  P3: B1 -> E2
  P4: D2 -> D3
  S4: D1 -> B2
 

## Is the resulting allocation feasible? Does it seem optimal?

The plan we got from the SDP model is feasible, as the rounding step ensures we stay under the two-person limit and keep everyone out of renovation zones. However, it isn't the absolute best way to do it if you want the fewest moves possible. The biggest problem is that the math solves each phase one by one without knowing what’s coming next . Since it can't see the future schedule, it often moves someone into an office that’s going to be closed for renovation in the very next phase, forcing them to move again right away . That's why this version ends up with about 62 moves, which is higher than the 36 moves we found when we planned everything out at once.

##